# Day 12 · Power · Study C: What AI Burns

Data centers next to the **power plants around them** (US EIA). For every site: total plant capacity within 50 km, the main fuel, and the gas share.
Nearby plants are a proxy for the local grid, not a data center's actual contract. The About text says so.

**Needs:** EIA Power Plants file in `data/raw/` (see README).

## 1 · Settings

In [ ]:
DAY      = 12
PROMPT   = "Power"
STUDY    = "C"
SLUG     = "power-ai"
NOTEBOOK = "day12_power_ai.ipynb"

TITLE     = "What AI Burns"
HIGHLIGHT = "AI"
SUBTITLE  = None        # None = filled in from the numbers in step 3
RADIUS_KM = 50          # how far to look for power plants around each site
MIN_MW    = 50          # hide small plants so the map stays readable
ABOUT     = ("Small colored dots are US power plants of 50 MW or more, colored by main fuel and sized by capacity "
             "(US EIA). For each data center and AI campus we add up plant capacity within 50 km and note the main fuel. "
             "That is the local grid around a site, not its power contract.")
CREDITS   = ["im3", "epoch", "eia"]

REGION   = "us48"
TOOL     = "mapbox"
MAPBOX_STYLE = "dark"
PROJECTION   = "globe"

## 2 · Load

In [ ]:
import pandas as pd
import mapkit as mk
from mapkit import ai, publish

dcs = ai.clip(ai.master(), REGION)
ex, big = dcs[dcs.kind == "existing"], dcs[dcs.kind == "ai_campus"]
print(f"{len(ex):,} existing data centers · {len(big)} AI campuses")
plants = ai.load_power_plants(min_mw=MIN_MW)
print(f"{len(plants):,} power plants of {MIN_MW} MW or more")
plants["fuel"].value_counts()

## 3 · Analysis: what powers the grid around each site?

In [ ]:
dcs = ai.power_nearby(dcs, plants, km=RADIUS_KM)
cols = [f"mw_within_{RADIUS_KM}km", f"main_fuel_{RADIUS_KM}km", f"gas_share_{RADIUS_KM}km"]
ai.update_master(dcs, cols, about=f"Power plant capacity, main fuel and gas share within {RADIUS_KM} km",
                 source="US EIA power plants", day=DAY)

ex, big = dcs[dcs.kind == "existing"], dcs[dcs.kind == "ai_campus"]
main = big[f"main_fuel_{RADIUS_KM}km"]
gas_main = (main == "Gas").sum()
gas_pct = round(100 * gas_main / max(len(big), 1))

print(f"AI campuses where gas is the main fuel within {RADIUS_KM} km: {gas_main} of {len(big)} ({gas_pct}%)")
gs = big[f"gas_share_{RADIUS_KM}km"]
print(f"Average gas share around AI campuses: {gs.mean():.0f}%" if gs.notna().any() else "No plants found near any AI campus")
print("\nMain fuel around AI campuses:")
print(main.value_counts().to_string())

if SUBTITLE is None:
    SUBTITLE = f"Gas is the main power source around {gas_pct}% of the new AI campuses"
print("\nSubtitle:", SUBTITLE)

plants["mw_label"] = plants["mw"].round(0)
overlays = [mk.Layer(plants, f"Power plants ({MIN_MW} MW+)", column="fuel", order=ai.FUEL_ORDER,
                     colors=ai.FUEL_COLORS, size_column="mw", radius=3, opacity=0.75, tooltip="plant",
                     popup={"fuel": "Fuel", "mw_label": "Capacity (MW)"})]
extra_popup = {f"main_fuel_{RADIUS_KM}km": f"Main fuel within {RADIUS_KM} km",
               f"gas_share_{RADIUS_KM}km": "Gas share nearby (%)"}

## 4 · Map

In [ ]:
layers = overlays + ai.base_layers(dcs, extra_popup=extra_popup)    # overlay underneath, the cast on top
bounds = ai.REGIONS[REGION] if isinstance(REGION, str) else None
credits = mk.style.credits(*CREDITS)

if TOOL == "mapbox":
    m = mk.mapbox(layers, title=TITLE, subtitle=SUBTITLE, about=ABOUT, sources=credits, series="ai",
                  bounds=bounds, style=MAPBOX_STYLE, projection=PROJECTION, highlight=HIGHLIGHT)
elif TOOL == "folium":
    m = mk.interactive(layers, day=DAY, prompt=PROMPT, title=TITLE, subtitle=SUBTITLE, sources=credits,
                       series="ai", bounds=bounds)
else:
    m = mk.static(layers, day=DAY, prompt=PROMPT, title=TITLE, subtitle=SUBTITLE, sources=credits,
                  series="ai", bounds=bounds, crs="EPSG:5070")
m if TOOL == "folium" else None

## 5 · Export + caption numbers

In [ ]:
publish.save(m, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE, subtitle=SUBTITLE, series="ai", study=STUDY,
             tool={"mapbox": "Python + Mapbox", "folium": "Python + Folium", "static": "Python"}[TOOL],
             notebook=NOTEBOOK, credits=mk.style.credits(*CREDITS, full=True))

print(f"""
CAPTION NUMBERS
- gas is the main fuel within {RADIUS_KM} km of {gas_main} of {len(big)} AI campuses ({gas_pct}%)
- main fuel around AI campuses: {", ".join(f"{k} {v}" for k, v in main.value_counts().items())}
""")

# ArcGIS Pro hero version? uncomment:
# publish.to_arcgis({"datacenters": dcs, "power_plants": plants}, day=DAY)